## TAREA 2 - PROBLEMAS DE CLASIFICACIÓN
#### *Machine Learning*

**Presentado por:**
- Violeta Isabella España Bolaños
- Sebastian Giraldo Acosta
- David Vergara
- Luis Murillo
- Juan David Velasco
---

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score

SEMILLA = 42
np.random.seed(SEMILLA)

### 0. Selección de Dataset

Para el desarrollo de la Tarea 2 seleccionamos el dataset *"Estimation of Obesity Levels Based on Eating Habits and Physical Condition"*, publicado en el repositorio UCI. Reúne 2.111 registros de personas de Colombia, México y Perú, con 17 variables sobre sus datos físicos, hábitos de alimentación, actividad física y antecedentes familiares. Cada persona está clasificada en uno de siete niveles de peso, desde peso insuficiente hasta obesidad tipo III.

In [21]:
# En Colab/entorno nuevo: !pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

# fetch dataset 
estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition = fetch_ucirepo(id=544) 
  
# data (as pandas dataframes) 
X = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.features 
y = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.targets

In [22]:
dataset = pd.concat([X, y], axis=1)
print("Dimensiones:", dataset.shape)
dataset.head()

Dimensiones: (2111, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [23]:
duplicados = dataset.duplicated().sum()
print("Cantidad de duplicados:", duplicados)

Cantidad de duplicados: 24


In [24]:
dataset = dataset.drop_duplicates()
print("Duplicados restantes:", dataset.duplicated().sum())

Duplicados restantes: 0


#### Codificación de variables categóricas de entrada

In [25]:
data = dataset.copy()

# 1. Codificación de variables binarias
data["Gender"] = data["Gender"].map({"Female": 0, "Male": 1})

for col in ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]:
    data[col] = data[col].map({"no": 0, "yes": 1})


# 2. Codificación de variables ordinales
for col in ["FCVC", "NCP", "CH2O", "FAF", "TUE"]:
    data[col] = data[col].round().astype(int)

orden_frecuencia = {
    "no": 0,
    "Sometimes": 1,
    "Frequently": 2,
    "Always": 3
}

data["CAEC"] = data["CAEC"].map(orden_frecuencia)

# Agrupar "Always" con "Frequently" porque solo hay un registro
data["CALC"] = data["CALC"].replace("Always", "Frequently")
data["CALC"] = data["CALC"].map(orden_frecuencia)


# 3. Codificación de variables nominales
data["MTRANS"] = data["MTRANS"].replace({
    "Bike": "Two_Wheels",
    "Motorbike": "Two_Wheels"
})

data = pd.get_dummies(
    data,
    columns=["MTRANS"],
    prefix="MTRANS",
    dtype=int
)


# 4. Codificación de la variable objetivo
niveles = {
    "Insufficient_Weight": 0,
    "Normal_Weight": 1,
    "Overweight_Level_I": 2,
    "Overweight_Level_II": 3,
    "Obesity_Type_I": 4,
    "Obesity_Type_II": 5,
    "Obesity_Type_III": 6
}

data["NObeyesdad"] = data["NObeyesdad"].map(niveles)

nombres_clases = {v: k for k, v in niveles.items()}


# 5. Separación de variables predictoras y objetivo
X = data.drop(columns=["NObeyesdad"])
y = data["NObeyesdad"]


# 6. Selección de variables que requieren escalado
columnas_numericas = [
    "Age", "Height", "Weight",
    "FCVC", "NCP", "CH2O",
    "FAF", "TUE", "CAEC", "CALC"
]


# 7. Configuración del preprocesador
preprocesador = ColumnTransformer(
    transformers=[
        ("escalado", StandardScaler(), columnas_numericas)
    ],
    remainder="passthrough"
)


# 8. Verificación de la transformación
print(f"Dimensiones: {data.shape}")
print(f"Nulos totales: {data.isna().sum().sum()}")
print(f"Columnas no numéricas: {data.select_dtypes(exclude='number').columns.tolist()}")

print("\nValores únicos por columna (excepto continuas):")

for col in data.columns.drop(["Age", "Height", "Weight"]):
    print(f"  {col:32s} {sorted(data[col].unique().tolist())}")

data.head()

Dimensiones: (2087, 20)
Nulos totales: 0
Columnas no numéricas: []

Valores únicos por columna (excepto continuas):
  Gender                           [0, 1]
  family_history_with_overweight   [0, 1]
  FAVC                             [0, 1]
  FCVC                             [1, 2, 3]
  NCP                              [1, 2, 3, 4]
  CAEC                             [0, 1, 2, 3]
  SMOKE                            [0, 1]
  CH2O                             [1, 2, 3]
  SCC                              [0, 1]
  FAF                              [0, 1, 2, 3]
  TUE                              [0, 1, 2]
  CALC                             [0, 1, 2]
  NObeyesdad                       [0, 1, 2, 3, 4, 5, 6]
  MTRANS_Automobile                [0, 1]
  MTRANS_Public_Transportation     [0, 1]
  MTRANS_Two_Wheels                [0, 1]
  MTRANS_Walking                   [0, 1]


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,NObeyesdad,MTRANS_Automobile,MTRANS_Public_Transportation,MTRANS_Two_Wheels,MTRANS_Walking
0,0,21.0,1.62,64.0,1,0,2,3,1,0,2,0,0,1,0,1,0,1,0,0
1,0,21.0,1.52,56.0,1,0,3,3,1,1,3,1,3,0,1,1,0,1,0,0
2,1,23.0,1.80,77.0,1,0,2,3,1,0,2,0,2,1,2,1,0,1,0,0
3,1,27.0,1.80,87.0,0,0,3,3,1,0,2,0,2,0,2,2,0,0,0,1
4,1,22.0,1.78,89.8,0,0,2,1,1,0,2,0,0,0,1,3,0,1,0,0


#### División del Dataset

In [26]:
# Separar características y variable objetivo
X = data.drop(columns=["NObeyesdad"])
y = data["NObeyesdad"]

# 70% entrenamiento y 30% restante
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEMILLA, stratify=y)

# 20% validación y 10% prueba
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=SEMILLA, stratify=y_temp)

### 1. Regularización L1 y L2 en tareas de clasificación

La regularización es una técnica utilizada en los modelos de clasificación para controlar su complejidad y reducir el sobreajuste (overfitting), evitando que aprendan excesivamente los patrones específicos de los datos de entrenamiento. Para ello, incorpora un término de penalización a la función de pérdida, que limita la magnitud de los pesos del modelo y favorece su capacidad de generalización ante datos nuevos.

Existen dos enfoques principales:

- Regularización L1 (Lasso): penaliza la suma de los valores absolutos de los pesos. Puede llevar algunos de ellos exactamente a cero, lo que permite reducir la cantidad de características utilizadas y favorece la selección automática de variables.

- Regularización L2 (Ridge): penaliza la suma de los cuadrados de los pesos. Reduce progresivamente su magnitud, evitando valores excesivamente grandes, pero generalmente conserva todas las características del modelo.

In [28]:
# Configración de hiperparámetros
# Valores de C que se evaluarán
# C pequeño = regularización más fuerte
# C grande = regularización más débil
parametros = {
    "clasificador__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

# Validación cruzada estratificada de 5 particiones
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEMILLA
)

In [64]:
modelo_L1 = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        penalty="l1",       # L1: favorece que algunos coeficientes sean exactamente cero
        solver="saga",      # Solver compatible con regularización L1 y clasificación multiclase
        max_iter=5000,      # Mayor número de iteraciones para favorecer la convergencia
        random_state=SEMILLA
    ))
])

busqueda_L1 = GridSearchCV(
    estimator=modelo_L1,
    param_grid=parametros,
    scoring="f1_macro",    # Evalúa el rendimiento promedio de las siete clases
    cv=cv,
    n_jobs=-1,
    refit=True
)

busqueda_L1.fit(X_train, y_train);

/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and u

In [76]:
modelo_L2 = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        penalty="l2",       # L2: reduce la magnitud de los coeficientes
        solver="newton-cholesky",      # Mejor F1 newton-cholesky - Para Comparar con L1 Saga
        max_iter=5000,      # Mayor número de iteraciones para favorecer la convergencia
        random_state=SEMILLA
    ))
])

busqueda_L2 = GridSearchCV(
    estimator=modelo_L2,
    param_grid=parametros,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True
)

busqueda_L2.fit(X_train, y_train);

/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/

#### Comparación F1 - Regularización L1 y L2

In [71]:
print("RESULTADOS DE LA BÚSQUEDA DE HIPERPARÁMETROS")

print("\nRegularización L1")
print("Mejor C:", busqueda_L1.best_params_["clasificador__C"])
print("Mejor F1-score macro:", busqueda_L1.best_score_)

print("\nRegularización L2")
print("Mejor C:", busqueda_L2.best_params_["clasificador__C"])
print("Mejor F1-score macro:", busqueda_L2.best_score_)

RESULTADOS DE LA BÚSQUEDA DE HIPERPARÁMETROS

Regularización L1
Mejor C: 10
Mejor F1-score macro: 0.9563334596540356

Regularización L2
Mejor C: 100
Mejor F1-score macro: 0.943826941534158


In [72]:
mejor_L1 = busqueda_L1.best_estimator_
mejor_L2 = busqueda_L2.best_estimator_

#### Comparación métricas con Regularización y sin Regularización

In [73]:
modelo_sin_regularizacion = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        penalty=None,             # Sin regularización
        solver="saga", # Mismo optimizador utilizado en L1
        max_iter=5000
    ))
])

modelo_sin_regularizacion.fit(X_train, y_train);

/home/violeta/miniconda3/envs/Desarrollo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [74]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Predicciones
pred_sin_reg = modelo_sin_regularizacion.predict(X_val)
pred_L1 = mejor_L1.predict(X_val)
pred_L2 = mejor_L2.predict(X_val)

# Comparación de métricas
resultados = pd.DataFrame({
    "Modelo": ["Sin regularización", "L1", "L2"],
    "Accuracy": [
        accuracy_score(y_val, pred_sin_reg),
        accuracy_score(y_val, pred_L1),
        accuracy_score(y_val, pred_L2)
    ],
    "Precision macro": [
        precision_score(y_val, pred_sin_reg, average="macro"),
        precision_score(y_val, pred_L1, average="macro"),
        precision_score(y_val, pred_L2, average="macro")
    ],
    "Recall macro": [
        recall_score(y_val, pred_sin_reg, average="macro"),
        recall_score(y_val, pred_L1, average="macro"),
        recall_score(y_val, pred_L2, average="macro")
    ],
    "F1-score macro": [
        f1_score(y_val, pred_sin_reg, average="macro"),
        f1_score(y_val, pred_L1, average="macro"),
        f1_score(y_val, pred_L2, average="macro")
    ]
})

resultados

,Modelo,Accuracy,Precision macro,Recall macro,F1-score macro
0,Sin regularización,0.949761,0.947942,0.948582,0.948130
1,L1,0.954545,0.953109,0.953243,0.953110
2,L2,0.937799,0.936006,0.936399,0.935999


**Observaciones**

El mejor comportamiento se observó en el modelo con regularización L1, el cual obtuvo un mejor desempeño en las métricas de evaluación en comparación con los modelos L2 y sin regularización. Esto sugiere que, para este conjunto de datos, la penalización L1 permitió reducir el peso de algunas variables predictoras e incluso llevar ciertos coeficientes a cero, favoreciendo un modelo más sencillo, al eliminar la influencia de características que aportaban poco a la clasificación.

Por otro lado, el modelo con regularización L2 presentó una reducción en todas las métricas evaluadas, incluso en comparación con el modelo sin regularización. Esto indica que, bajo la configuración utilizada, la penalización aplicada pudo haber limitado demasiado los valores de los coeficientes, afectando la capacidad del modelo para distinguir entre las clases. En este caso, la regularización no representó una mejora en el desempeño.

### 2. Pérdida de entropía cruzada